In [ ]:
from openmm.app import *
from openmm import *
from openmm.unit import *
from sys import stdout
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
from openff.toolkit import Molecule
from openmmforcefields.generators import SystemGenerator
from matplotlib.patheffects import withStroke


class InteractionEnergy:
    def __init__(self, topology, system, modeller, context):
        self.topology = topology
        self.system = system
        self.modeller = modeller
        self.context = context
        self.protein_atoms = self.get_protein_atoms()
        self.protein_residues = [residue.index for residue in topology.residues() if residue.name not in ('HOH', 'CL', 'NA')]
        self.atom_charges = self.init_params()

    def get_protein_atoms(self):
        """Fetch all atoms residing in the protein (excluding solvent)."""
        protein_atoms = []
        for residue in self.topology.residues():
            if residue.name not in ('HOH', 'CL', 'NA'):  # Ignore solvent
                for atom in residue.atoms():
                    protein_atoms.append(atom.index)
        return protein_atoms

    def get_residue_number_of_atom(self, atom_index):
        """Get the residue number for a given atom by its index."""
        for atom in self.topology.atoms():
            if atom.index == atom_index:
                return atom.residue.index
        raise ValueError(f"Atom with index {atom_index} not found in topology.")

    def init_params(self):
        """Initialize parameters and modify NonbondedForce for energy decomposition."""
        atom_charges = {}
        for force in self.system.getForces():
            if isinstance(force, NonbondedForce):
                force.setForceGroup(0)
                force.addGlobalParameter("solvent_coul_scale", 0)
                force.addGlobalParameter("solvent_lj_scale", 0)
                for i in self.protein_residues:
                    force.addGlobalParameter(f"residue{i}_coul_scale", 0)
                    force.addGlobalParameter(f"residue{i}_lj_scale", 0)

                for atom in range(force.getNumParticles()):
                    charge, sigma, epsilon = force.getParticleParameters(atom)

                    # Set default interaction to zero
                    force.setParticleParameters(atom, 0, 0, 0)
                    if atom in self.protein_atoms:
                        atom_charges[atom] = charge.value_in_unit(elementary_charge)
                        residue_number = self.get_residue_number_of_atom(atom)
                        force.addParticleParameterOffset(f"residue{residue_number}_coul_scale", atom, charge, 0, 0)
                        force.addParticleParameterOffset(f"residue{residue_number}_lj_scale", atom, 0, sigma, epsilon)
                        
                    elif atom in solvent:
                        force.addParticleParameterOffset("solvent_coul_scale", atom, charge, 0, 0)
                        force.addParticleParameterOffset("solvent_lj_scale", atom, 0, sigma, epsilon)

                for i in range(force.getNumExceptions()):
                    p1, p2, chargeProd, sigma, epsilon = force.getExceptionParameters(i)
                    force.setExceptionParameters(i, p1, p2, 0, 0, 0)

            else:
                force.setForceGroup(2)
                
        return atom_charges

    def reset_context(self):
        """Reset context parameters."""
        for i in self.protein_residues:
            self.context.setParameter(f"residue{i}_coul_scale", 0)
            self.context.setParameter(f"residue{i}_lj_scale", 0)
        self.context.setParameter(f"solvent_coul_scale", 0)
        self.context.setParameter(f"solvent_lj_scale", 0)

    def get_energy_residue_residue(self, residue_number1, residue_number2, force_type, residue1_scale, residue2_scale):
        """Get energy for residue-residue interaction."""
        if residue_number1 == residue_number2:
            return 0
        else:
            self.context.setParameter(f"solvent_{force_type}_scale", 0)
            for i in self.protein_residues:
                self.context.setParameter(f"residue{i}_{force_type}_scale", 0)
            self.context.setParameter(f"residue{residue_number1}_{force_type}_scale", residue1_scale)
            self.context.setParameter(f"residue{residue_number2}_{force_type}_scale", residue2_scale)

            return self.context.getState(getEnergy=True, groups={0}).getPotentialEnergy().value_in_unit(kilojoule_per_mole)

    def get_energy_residue_rest(self, residue_number, force_type, residue, rest, solvent):
        """Get energy for residue-rest interaction."""
        for i in self.protein_residues:
            self.context.setParameter(f"residue{i}_{force_type}_scale", rest)
        self.context.setParameter(f"residue{residue_number}_{force_type}_scale", residue)
        self.context.setParameter(f"solvent_{force_type}_scale", solvent)

        return self.context.getState(getEnergy=True, groups={0}).getPotentialEnergy().value_in_unit(kilojoule_per_mole)

    def get_interaction_energy_residue_residue(self, residue_number1, residue_number2):
        """Returns the (lj, coul) interaction energy tuple for residue-residue interaction."""
        self.reset_context()
        lj = self.get_energy_residue_residue(residue_number1, residue_number2, "lj", 1, 1) - \
             self.get_energy_residue_residue(residue_number1, residue_number2, "lj", 0, 1) - \
             self.get_energy_residue_residue(residue_number1, residue_number2, "lj", 1, 0)
        coul = self.get_energy_residue_residue(residue_number1, residue_number2, "coul", 1, 1) - \
               self.get_energy_residue_residue(residue_number1, residue_number2, "coul", 0, 1) - \
               self.get_energy_residue_residue(residue_number1, residue_number2, "coul", 1, 0)
        return lj, coul

    def get_interaction_energy_to_protein(self, residue_number):
        """Returns the (lj, coul) interaction energy tuple for a residue's interaction with the protein."""
        self.reset_context()
        lj = self.get_energy_residue_rest(residue_number, "lj", 1, 1, 0) - \
             self.get_energy_residue_rest(residue_number, "lj", 1, 0, 0) - \
             self.get_energy_residue_rest(residue_number, "lj", 0, 1, 0)
        coul = self.get_energy_residue_rest(residue_number, "coul", 1, 1, 0) - \
               self.get_energy_residue_rest(residue_number, "coul", 1, 0, 0) - \
               self.get_energy_residue_rest(residue_number, "coul", 0, 1, 0)
        return lj, coul
    
    def get_residue_positions(self, residue_to_scan):
        """Fetch the residue positions and atoms for scanning."""
        positions = self.context.getState(getPositions=True).getPositions()
        residue_centroids = []
        scan_residue_atoms = []
        scan_residue_index = []

        for residue in self.topology.residues():
            if residue.name not in ("HOH", "NA", "CL"):
                residue_pos = []
                for atom in residue.atoms():
                    # If atom in residue we want to scan
                    if residue.index == residue_to_scan:
                        scan_residue_atoms.append(positions[atom.index].value_in_unit(angstrom))
                        scan_residue_index.append((atom.index, self.atom_charges[atom.index]))

                    # Either way we need to find residue center
                    residue_pos.append(positions[atom.index].value_in_unit(angstrom))

                residue_pos = np.array(residue_pos)
                x_mean = np.mean(residue_pos[:, 0])
                y_mean = np.mean(residue_pos[:, 1])
                z_mean = np.mean(residue_pos[:, 2])
                residue_centroids.append((x_mean, y_mean, z_mean))

        # Convert to arrays
        residue_centroids = np.array(residue_centroids)
        scan_residue_atoms = np.array(scan_residue_atoms)
        scan_residue_index = np.array(scan_residue_index)

        # Correct the scan_residue_atoms to use 0,0,0 coordinate at ligand center
        scan_residue_atoms -= residue_centroids[residue_to_scan]

        return residue_centroids, scan_residue_atoms, scan_residue_index

    def get_residue_residue_dist(self, residue_centroids, residue1, residue2):
        """Calculate distance between residue centroids."""
        centroid_1 = residue_centroids[residue1]
        centroid_2 = residue_centroids[residue2]
        return np.linalg.norm(centroid_1 - centroid_2)

    def residue_residue_map(self, residue_to_scan, residue_centroids):
        """Generate residue-residue interaction map."""
        interactions_list = []

        for residue in self.topology.residues():
            if residue.name not in ("HOH", "NA", "CL"):
                lj, coul = self.get_interaction_energy_residue_residue(residue_to_scan, residue.index)
                distance = self.get_residue_residue_dist(residue_centroids, residue_to_scan, residue.index)
                interactions_list.append([residue.name + str(residue.index + 1), residue.index, lj, coul, distance])

        interaction_map = pd.DataFrame(interactions_list, columns=["residue_name", "residue_index", "LJ_energy", "Coul_energy", "distance"])
        return interaction_map

    def interaction_plotter(self, interaction_map, threshold):
        """Plot the interaction energies (LJ, Coul) and distances."""
        interactions_filtered = interaction_map[(abs(interaction_map["LJ_energy"]) > threshold) | (abs(interaction_map["Coul_energy"]) > threshold)]
        x_array = np.arange(0, len(interactions_filtered), step=1)
        print(f"interaction table \n {interactions_filtered}")

        plt.figure(figsize=(14, 4))
        plt.bar(x_array + 0.2, interactions_filtered["Coul_energy"], 0.4, label="Coul", color="darkblue")
        plt.bar(x_array - 0.2, interactions_filtered["LJ_energy"], 0.4, label="LJ", color="red")
        plt.bar(x_array, interactions_filtered["distance"], 0.4, label="Dist", color="darkgreen")

        # Add residue labels
        for i, residue_name in enumerate(interactions_filtered["residue_name"]):
            plt.text(i, 0, residue_name, ha="center", color="white", path_effects=[ withStroke(linewidth=1.5, foreground='black')])
            
        plt.title(f"interaction plotter at {threshold} kJ/mol")    
        plt.xlabel("Residue number")
        plt.ylabel("kJ/mol")
        plt.xlim(-0.4, len(x_array) + 0.4)
        plt.axhline(0, color='black', linewidth=0.8)  # Draw a horizontal line at y=0
        plt.legend(loc="upper right")
        plt.show()

    def electrostatic_plotter(self, interaction_map, threshold):
        """Plot the electrostatic interactions with distance."""
        interactions_filtered = interaction_map[interaction_map["distance"] < threshold]
        print(f"electrostatic table \n {interactions_filtered}")
        x_array = np.arange(0, len(interactions_filtered), step=1)

        plt.figure(figsize=(14, 4))
        plt.bar(x_array + 0.2, interactions_filtered["Coul_energy"] * interactions_filtered["distance"]**2 / (1.60217663 * 100), 0.4, label="q1*q2", color="darkblue")
        plt.bar(x_array, interactions_filtered["distance"], 0.4, label="Dist", color="darkgreen")

        # Add residue labels
        for i, residue_name in enumerate(interactions_filtered["residue_name"]):
            plt.text(i, 0, residue_name, ha="center", color="white", path_effects=[ withStroke(linewidth=1.5, foreground='black')])
        
        plt.title(f"Electrostatic plotter at {threshold} Å")   
        plt.xlabel("Residue number")
        plt.ylabel("q1*q2 or Å")
        plt.xlim(-0.4, len(x_array) + 0.4)
        plt.axhline(0, color='black', linewidth=0.8)  # Draw a horizontal line at y=0
        plt.legend(loc="upper right")
        plt.show()

    def get_distance_weighted_q1(self, residue_centroids, scan_residue_atoms, residue_to_scan):
        """Calculate distance-weighted q1 charge interaction."""
        q1 = []

        for residue in range(len(residue_centroids)):
            if residue == residue_to_scan:
                q1.append(0)
            else:
                q_atoms = 0
                for atom in range(len(scan_residue_atoms)):
                    distance = np.linalg.norm(scan_residue_atoms[atom] + residue_centroids[residue_to_scan] - residue_centroids[residue])
                    q_atoms += scan_residue_index[atom, 1] / distance**2
                q1.append(q_atoms)

        return np.array(q1)

    def q1_plot(self, interaction_map, dist_threshold):
        """Plot q1 interaction data."""

        # Scaling factor from unit charge^2/Å^2 to kJ/mol
        scale = (10 ** -3) * (6.022 * 10 ** 23) * (1.602 * 10 ** -19) ** 2 * 10 ** 20

        interactions_filtered = interaction_map[interaction_map["distance"] < dist_threshold]
        x_array = np.arange(0, len(interactions_filtered), step=1)

        interactions_filtered.loc[:, "q1"] = interactions_filtered["q1"] * -scale

        plt.figure(figsize=(14, 4))
        plt.bar(x_array, interactions_filtered["q1"], 0.4, label="q1", color="darkblue")

        print(f"q1 table \n {interactions_filtered}")

        # Add residue labels
        for i, residue_name in enumerate(interactions_filtered["residue_name"]):
            plt.text(i, 0, residue_name, ha="center", color="white", path_effects=[ withStroke(linewidth=1.5, foreground='black')])
        
        plt.title(f"q1 plot at {dist_threshold} Å")   
        plt.xlabel("Residue number")
        plt.ylabel("kJ/mol for interaction with a -1 unit charge")
        plt.xlim(-0.4, len(x_array) + 0.4)
        plt.axhline(0, color='black', linewidth=0.8)  # Draw a horizontal line at y=0
        plt.legend(loc="upper right")
        plt.show()



# MAIN code

pdb1 = "5tbm-chain_equillibrated"
pdb = PDBFile(pdb1 + ".pdb")

# Identify solvent and protein atoms
solvent = set([a.index for a in pdb.topology.atoms() if a.residue.name in ('HOH', 'CL', 'NA')])

ligand_mol = Molecule.from_file("PT2385.sdf")

# Specify the forcefield
forcefield_kwargs = {'constraints': HBonds, 'rigidWater': True, 'removeCMMotion': False}
system_generator = SystemGenerator(
    forcefields=['amber14-all.xml', 'amber14/tip3pfb.xml'],
    small_molecule_forcefield='openff-2.2.1.offxml',
    molecules=[ligand_mol],
    forcefield_kwargs=forcefield_kwargs)

modeller = Modeller(pdb.topology, pdb.positions)

# Create the system using the SystemGenerator
system = system_generator.create_system(modeller.topology)

# Initialize InteractionEnergy class before creating the Context
interaction_energy = InteractionEnergy(pdb.topology, system, modeller, None)

# Create integrator and context
integrator = LangevinMiddleIntegrator(300 * kelvin, 1 / picosecond, 0.004 * picoseconds)

context = Context(system, integrator)
context.setPositions(modeller.positions)

# Now, initialize the context parameterization
interaction_energy.context = context  # Assign the context after it is created
interaction_energy.init_params()  # Call init_params() after the context has been created

# Iterate over frames in the PDB
#for frame_index, positions in enumerate(pdb.positions):
    # Update the simulation context with the new frame positions
#    simulation.context.setPositions(positions)
    
#user input
residue_to_scan = 113

#0 index since everything not directlly shown to user is zero indexed
residue_to_scan -= 1

# Get residue positions and interaction data
residue_centroids, scan_residue_atoms, scan_residue_index = interaction_energy.get_residue_positions(residue_to_scan)

# Generate residue-residue interaction map
interaction_map = interaction_energy.residue_residue_map(residue_to_scan, residue_centroids)

# Plot interaction data
interaction_energy.interaction_plotter(interaction_map, 17)
interaction_energy.electrostatic_plotter(interaction_map, 8)

# Calculate and add q1 charge interaction to the map
q1 = interaction_energy.get_distance_weighted_q1(residue_centroids, scan_residue_atoms, residue_to_scan)
interaction_map["q1"] = q1

# Plot q1 interaction
interaction_energy.q1_plot(interaction_map, 8)

# Example of interaction energy calculation
lj, coul = interaction_energy.get_interaction_energy_to_protein(residue_to_scan)
print(f"Residue {residue_to_scan + 1} interaction with protein: LJ = {lj}, Coulomb = {coul}")


OpenMMException: NonbondedForce: The cutoff distance cannot be greater than half the periodic box size.  For more information, see https://github.com/openmm/openmm/wiki/Frequently-Asked-Questions#boxsize